In [1]:
# Importar las bibliotecas necesarias
import LanusStats as ls
from LanusStats.functions import get_available_pages, get_available_leagues, get_available_season_for_leagues
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuraciones de visualización
pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

# Ver las fuentes disponibles
pages = get_available_pages()
print(f"Fuentes disponibles: {pages}")

# Explorar ligas disponibles en cada fuente
for page in pages:
    print(f"\nLigas disponibles en {page}:")
    try:
        leagues = get_available_leagues(page)
        for league in leagues:
            if 'MX' in league or 'Mexico' in league:
                print(f"- {league}")
    except Exception as e:
        print(f"Error: {e}")

Fuentes disponibles: dict_keys(['Fbref', 'Sofascore', '365Scores', 'Fotmob', 'Transfermarkt', 'DataFactory'])

Ligas disponibles en Fbref:
- Liga MX

Ligas disponibles en Sofascore:
- Mexico LigaMX Apertura
- Mexico LigaMX Clausura

Ligas disponibles en 365Scores:

Ligas disponibles en Fotmob:
- Liga MX

Ligas disponibles en Transfermarkt:

Ligas disponibles en DataFactory:


In [2]:
# Seleccionamos una página (en este caso, "Fbref") y obtenemos las ligas disponibles
page = "Fbref"
# La función get_available_leagues requiere pasar un ejemplo de liga y temporada, ya que internamente se usa para extraer la estructura.
# Usaremos "Argentina Copa de la Liga" como ejemplo (ya que está disponible) para poder listar las ligas.
# IMPORTANTE: get_available_leagues devuelve la lista de ligas de la página utilizada en la llamada a la función get_possible_leagues.
leagues_fbref = get_available_leagues(page)
print(f"\nLigas disponibles en {page}:")
for league in leagues_fbref:
    print("-", league)


Ligas disponibles en Fbref:
- Copa de la Liga
- Primera Division Argentina
- Primera Division Uruguay
- Brasileirao
- Brasileirao B
- Primera Division Colombia
- Primera Division Chile
- Primera Division Peru
- Primera Division Venezuela
- Primera Division Ecuador
- Primera Division Bolivia
- Primera Division Paraguay
- Brasileirao F
- MLS
- USL Championship
- Premier League
- La Liga
- Ligue 1
- Bundesliga
- Serie A
- Big 5 European Leagues
- Danish Superliga
- Eredivise
- Primeira Liga Portugal
- Copa America
- Euros
- Saudi League
- EFL Championship
- La Liga 2
- Belgian Pro League
- Challenger Pro League
- 2. Bundesliga
- Ligue 2
- Serie B
- J1 League
- NSWL
- Wowens Super League
- Liga F
- Premier Division South Africa
- Champions League
- Europa League
- Conference League
- Copa Libertadores
- Liga MX


In [3]:
from LanusStats.functions import get_possible_leagues

# Lista de páginas (fuentes) que deseas revisar
pages = ['Fbref', 'Sofascore', '365Scores', 'Fotmob', 'Transfermarkt', 'DataFactory']

# Recorremos cada página y buscamos la información de "Liga MX"
for page in pages:
    print(f"\n=== Página: {page} ===")
    
    try:
        # Llamamos a get_possible_leagues() para obtener el diccionario global.
        # Le pasamos "Liga MX" como liga y None como temporada.
        # Esto devolverá la estructura global, pero se indexa [page] para quedarnos con el dict propio de esa página.
        leagues_for_page = get_possible_leagues("Liga MX", None, page)[page]

        # leagues_for_page ya debería ser un diccionario de ligas con su id, slug y seasons
        # Recorremos cada liga definida en esta página
        for league_name, league_data in leagues_for_page.items():
            # Filtramos para encontrar nombres que contengan "MX" o "Mexico"
            if "MX" in league_name or "Mexico" in league_name:
                # Obtenemos las temporadas (si existe, de lo contrario una lista vacía)
                seasons = league_data.get('seasons', [])
                
                print(f"  Liga: {league_name}")
                print(f"    ID: {league_data.get('id')}")
                print(f"    Slug: {league_data.get('slug')}")
                print(f"    Temporadas disponibles: {seasons}")

    except Exception as e:
        print(f"  Error al obtener datos para {page}: {e}")


=== Página: Fbref ===
  Liga: Liga MX
    ID: 31
    Slug: Liga-MX
    Temporadas disponibles: {'2020-2021', '2019-2020', '2023-2024', '2018-2019', '2021-2022', '2022-2023', '2024-2025'}

=== Página: Sofascore ===
  Error al obtener datos para Sofascore: League Liga MX is not valid for any of the possible leagues ['Argentina Liga Profesional', 'Argentina Copa de la Liga Profesional', 'Argentina Primera Nacional', 'Brasileirão Série A', 'Bolivia Division Profesional', 'Chile Primera Division', 'Colombia Primera A Apertura', 'Colombia Primera A Clausura', 'Ecuador LigaPro', 'Mexico LigaMX Apertura', 'Mexico LigaMX Clausura', 'Peru Liga 1', 'Uruguay Primera Division', 'Venezuela Primera Division', 'World Cup', 'Euros', 'Copa America', 'Premier League', 'La Liga', 'Bundesliga', 'Serie A', 'Ligue 1', 'Copa Libertadores', 'Copa Sudamericana', 'MLS', 'Saudi Pro League', 'J1 League', 'NSWL', 'USL Championship', 'La Liga 2'].

=== Página: 365Scores ===
  Error al obtener datos para 365Scores: 

In [2]:
import random
import time
from LanusStats.functions import get_possible_leagues_for_page
from LanusStats.fotmob import FotMob
import pandas as pd

def test_random_season_fotmob():
    page = "Fotmob"
    league = "Liga MX"
    
    # Obtener la configuración para la liga en Fotmob
    try:
        leagues = get_possible_leagues_for_page(league, None, page)
        liga_info = leagues.get(league)
        if not liga_info:
            print(f"No se encontró la configuración para {league} en {page}.")
            return
    except Exception as ex:
        print("Error al obtener la configuración:", ex)
        return

    # Seleccionar una temporada al azar de la lista de temporadas disponibles
    seasons = liga_info.get("seasons")
    if not seasons:
        print("No hay temporadas definidas para la liga.")
        return

    season = random.choice(seasons)
    print("Probando FotMob para Liga MX, temporada seleccionada al azar:", season)
    
    # Crear instancia de FotMob y llamar al método para obtener la tabla de la temporada
    fotmob = FotMob()
    try:
        table_df = fotmob.get_season_tables(league, season)
        
        # Si el resultado es una lista (porque las tablas están split), intentamos extraer la tabla "all"
        if isinstance(table_df, list):
            try:
                # Se asume que el primer elemento de la lista es el contenedor de tablas
                # y se busca la tabla correspondiente a "all"
                table_data = table_df[0]["table"]["all"]
                table_df = pd.DataFrame(table_data)
                print("Se seleccionó la tabla 'all' de la respuesta split.")
            except Exception as e:
                print("Error al extraer la tabla 'all' de la respuesta:", e)
                return
        print("Tabla obtenida (primeras filas):")
        print(table_df.head())
    except Exception as e:
        print("Error al obtener la tabla para la temporada", season, ":", e)

def main():
    test_random_season_fotmob()
    time.sleep(3)

if __name__ == "__main__":
    main()


Probando FotMob para Liga MX, temporada seleccionada al azar: 2022/2023 - Clausura
This response has a list of two values, because the tables are split. If you save the list in a variable and then do variable[0]["table"] you will have all of the tables
Then just select one ["all", "home", "away", "form", "xg"] that exists and put it inside a pd.DataFrame()
Something like pd.DataFrame(variable[0]["table"]["all"])
Se seleccionó la tabla 'all' de la respuesta split.
Tabla obtenida (primeras filas):
         name  shortName    id                          pageUrl deduction  \
0   Monterrey  Monterrey  7849   /teams/7849/overview/monterrey      None   
1  CF America    América  6576  /teams/6576/overview/cf-america      None   
2      Chivas     Chivas  7807      /teams/7807/overview/chivas      None   
3      Toluca     Toluca  6618      /teams/6618/overview/toluca      None   
4     Pachuca    Pachuca  7848     /teams/7848/overview/pachuca      None   

  ongoing  played  wins  draws  loss